# 读取LigandMPNN的结果csv文件，按照设计的个数进行排序，取前十的序列构建Protenix的输入json文件

In [ ]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/template.json"

pdbs = [pdb_file.split('.')[0] for pdb_file in os.listdir('/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length') if pdb_file.endswith('.pdb')]

res = '020'
temperature = '0.2'


#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件


for pdb in pdbs:

    jobs = []
    
    with open(json_temple_path, 'r') as file:
        tmpl = json.load(file)

    path = f'./Output/{pdb}-merged-0.2-1_2/ligandmpnn_v_32_{res}_25'

    df = pd.read_csv(f'{path}/{pdb}_rescored_top10-autoregressive.csv')
    
    for _, row in df.iloc[:10].iterrows():
        name = row.iloc[0]
        seq_pro = row.iloc[1]
        seq_pep = row.iloc[2]

        job = json.loads(json.dumps(tmpl[0]))
        job['name'] = pdb + "_" + str(name)
        job['sequences'][0]['proteinChain']['sequence'] = seq_pro
        job['sequences'][1]['proteinChain']['sequence'] = seq_pep
        jobs.append(job.copy())


    with open(f'{path}/protenix_pred-filter1.json', 'w') as f:
        f.write(json.dumps(jobs, indent=4))

In [ ]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "template.json"

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]

res = '020'

filter = 12

#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件


for pdb in pdbs:

    jobs = []
    
    with open(json_temple_path, 'r') as file:
        tmpl = json.load(file)

    path = f'./Output/{pdb}-0.1T/ligandmpnn_v_32_{res}_25'

    df = pd.read_csv(f'{path}/{pdb}_rescored_top10-autoregressive.csv')

    msa_pro_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length/MSA/{pdb}'
    pairing_db = 'uniref100'
    for _, row in df.iloc[:10].iterrows():
        name = row.iloc[0]
        seq_pro = row.iloc[1]
        seq_pep = row.iloc[2]
        job_name = pdb + "_" + str(name)

        # 编写多肽的msa文件
        pep_base_dir = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Protenix/10000-0.1T/{pdb}' 
        msa_pep_path = f'{pep_base_dir}/predict_output-filter{filter}/{job_name}/msa_pep'
        os.makedirs(msa_pep_path, exist_ok=True)
        with open(f'{msa_pep_path}/pairing.a3m', 'w') as f:
            f.write('>query\n')
            f.write(seq_pep + '\n')
        os.system(f'cp {msa_pep_path}/pairing.a3m {msa_pep_path}/non_pairing.a3m')
        
        job = json.loads(json.dumps(tmpl[0]))
        job['name'] = job_name
        job['sequences'][0]['proteinChain']['sequence'] = seq_pro
        job['sequences'][0]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pro_path
        job['sequences'][0]['proteinChain']['msa']['pairing_db'] = pairing_db

        job['sequences'][1]['proteinChain']['sequence'] = seq_pep
        job['sequences'][1]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pep_path
        job['sequences'][1]['proteinChain']['msa']['pairing_db'] = pairing_db
        jobs.append(job.copy())


    with open(f'{pep_base_dir}/pred_filter{filter}.json', 'w') as f:
        f.write(json.dumps(jobs, indent=4))

### PepSet-passed-dimer_processed

In [ ]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "template.json"

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB_pep_plddt.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]


#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件


for pdb in pdbs:

    ori_pdb = pdb.split('_')[0]

    path = f'./LigandMPNN-Output-pred_dimer/2000-0.1T/{pdb}'
    for seed in ['seed42', 'seed43', 'seed44']:
        jobs = []                                     # 循环内清空jobs
        with open(json_temple_path, 'r') as file:     # 不能在循环外打开，否则后续seed43和seed44会包含前面循环的内容，即每一次循环需要重新读取template.json
            tmpl = json.load(file)
        df = pd.read_csv(f'{path}/{seed}/ranked_by_Max_Overall_Confidence.csv')

        ori_msa_pro_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/MSA_pro_all_pep_set_dimer/{ori_pdb}'
        pairing_db = 'uniref100'
        pep_base_dir = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/{pdb}'  # full path
        for _, row in df.iloc[:10].iterrows():
            name = row.iloc[0]
            seq_pro = row.iloc[1]
            seq_pep = row.iloc[2]
            job_name = pdb + "_" + str(name)

            # 编写多肽的msa文件
            msa = f'{pep_base_dir}/ligandmpnn_{seed}/{job_name}/msa'
            msa_pep_path = f'{msa}/msa_pep'
            msa_pro_path = f'{msa}/msa_pro'
            os.makedirs(msa_pep_path, exist_ok=True)
            os.makedirs(msa_pro_path, exist_ok=True)
            with open(f'{msa_pep_path}/pairing.a3m', 'w') as f:
                f.write('>query\n')
                f.write(seq_pep + '\n')
            os.system(f'cp {msa_pep_path}/pairing.a3m {msa_pep_path}/non_pairing.a3m')

            os.system(f'cp {ori_msa_pro_path}/pairing.a3m {msa_pro_path}/pairing.a3m')
            os.system(f'cp {ori_msa_pro_path}/non_pairing.a3m {msa_pro_path}/non_pairing.a3m')

            job = json.loads(json.dumps(tmpl[0]))
            job['name'] = job_name
            job['sequences'][0]['proteinChain']['sequence'] = seq_pro
            job['sequences'][0]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pro_path
            job['sequences'][0]['proteinChain']['msa']['pairing_db'] = pairing_db

            job['sequences'][1]['proteinChain']['sequence'] = seq_pep
            job['sequences'][1]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pep_path
            job['sequences'][1]['proteinChain']['msa']['pairing_db'] = pairing_db
            jobs.append(job.copy())


        with open(f'{pep_base_dir}/ligandmpnn_{seed}/pred.json', 'w') as f:
            f.write(json.dumps(jobs, indent=4))

# 对Protenix的预测指标进行分析

### 按照蛋白链对齐，计算多肽链骨架的RMSD

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer

filter = 12
res = '020'
temperature = '0.1'
pdbs = [pdb_file.split('.')[0] for pdb_file in os.listdir('/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length') if pdb_file.endswith('.pdb')]

pdbparser = PDBParser(QUIET=True)
mmcifparser = FastMMCIFParser(QUIET=True)

for pdb in pdbs:
    base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Protenix/10000-0.1T/{pdb}/predict_output-filter{filter}"
    merged_pdb_path = Path(f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs/{pdb}.pdb")
    if not os.path.exists(merged_pdb_path):
        raise FileNotFoundError(f"Merged PDB file not found: {merged_pdb_path}")
    
    structure_ref = pdbparser.get_structure('ref', merged_pdb_path)
    chain_L_ref = structure_ref[0]['L']
    ref_atoms = [atom for atom in chain_L_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]

    rows = []

    for root, dirs, files in os.walk(base_dir):
        seed = root.split("/")[-2]
        sample_name = root.split("/")[-3]

        candidates = []
        if not root.endswith("predictions"):
            continue
        
        metrics_by_id = {}

        for file in files:
            if file.endswith(".json"):
                sample_id = file.split("_")[-1].split(".")[0]
                json_path = os.path.join(root, file)
                if not os.path.exists(json_path):
                    continue
                df = pd.read_json(json_path)
                metrics_by_id.setdefault(sample_id, {}).update(
                    {
                        "plddt": round(float(df["plddt"].iloc[0]), 4),
                        "gpde": round(float(df["gpde"].iloc[0]), 4),
                        "ptm": round(float(df["ptm"].iloc[0]), 4),
                        "iptm": round(float(df["iptm"].iloc[0]), 4),
                        "ranking_score": round(float(df["ranking_score"].iloc[0]), 4),
                    }
                )

            elif file.endswith(".cif"):
                sample_id = file.split("_")[-1].split(".")[0]
                cif_path = os.path.join(root, file)
                if not os.path.exists(cif_path):
                    continue
                structure_pred = mmcifparser.get_structure('pred', cif_path)

                # 预测结构中的 L 链（原逻辑为 'B'）
                try:
                    chain_L_pred = structure_pred[0]['B']
                except KeyError:
                    # 若不存在 B 链，跳过
                    continue

                # 参考结构的蛋白链（选取第一个不为 'L' 的链）
                align_chain_ref = None
                for ch in structure_ref[0]:
                    if ch.id != 'L':
                        align_chain_ref = ch
                        break
                if align_chain_ref is None:
                    continue

                # 预测结构的蛋白链（选取第一个不为预测 L 链的链）
                align_chain_pred = None
                for ch in structure_pred[0]:
                    if ch.id != chain_L_pred.id:
                        align_chain_pred = ch
                        break
                if align_chain_pred is None:
                    continue

                # 取用于对齐的蛋白链骨架原子
                align_ref_atoms = [atom for atom in align_chain_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                align_pred_atoms = [atom for atom in align_chain_pred.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                if len(align_ref_atoms) == 0 or len(align_pred_atoms) == 0:
                    raise ValueError("No backbone atoms found for alignment.")
                if len(align_ref_atoms) != len(align_pred_atoms):
                    # 原子数不一致，无法对齐，跳过
                    raise ValueError("Backbone atom count mismatch for alignment.")

                # 计算对齐变换（基于蛋白链）
                sup = Superimposer()
                try:
                    sup.set_atoms(align_ref_atoms, align_pred_atoms)
                except Exception:
                    raise ValueError("Failed to set atoms for superimposer.")
                rot, tran = sup.rotran

                # 在该变换下计算 L 链骨架的 RMSD
                ref_L_atoms = [atom for atom in chain_L_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                pred_L_atoms = [atom for atom in chain_L_pred.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                if len(ref_L_atoms) == 0 or len(pred_L_atoms) == 0:
                    raise ValueError("No backbone atoms found for L chain.")
                if len(ref_L_atoms) != len(pred_L_atoms):
                    raise ValueError("Backbone atom count mismatch for L chain.")

                ref_coords = np.array([a.get_coord() for a in ref_L_atoms], dtype=float)
                pred_coords = np.array([a.get_coord() for a in pred_L_atoms], dtype=float)
                pred_coords_aligned = pred_coords @ rot + tran

                diffs = ref_coords - pred_coords_aligned
                scRMSD = float(np.sqrt(np.mean(np.sum(diffs**2, axis=1))))
                scRMSD = round(scRMSD, 4)
                metrics_by_id.setdefault(sample_id, {})["scRMSD"] = scRMSD

        for sample_id, data in metrics_by_id.items():
            required_keys = {"plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"}
            if not required_keys.issubset(data):
                continue
            candidates.append(
                (
                    sample_name,
                    seed,
                    sample_id,
                    data["plddt"],
                    data["gpde"],
                    data["ptm"],
                    data["iptm"],
                    data["ranking_score"],
                    data["scRMSD"],
                )
            )

        # flatten all candidate rows across directories
        rows.extend(candidates)

    result = pd.DataFrame(rows, columns=["complex", "seed", "id", "plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"])
    result.to_csv(f"{base_dir}/metrics_summary-filter{filter}.csv", index=False)

    # 过滤scRMSD大于1的结果，保存到新的df中
    result_filtered = result[result['scRMSD'] <= 2.0].reset_index(drop=True)
    result_filtered.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le2-filter{filter}.csv", index=False)
    result_filtered

# 调整plddt为多肽链的plddt

In [4]:
import os
import pandas as pd
import json

filters = [2, 3, 4, 5]

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet_3per_length/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

for filter in filters:
    for pdb in pdbs:
        base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/11000-0.1_0.2T/{pdb}/predict_output-filter{filter}"
        metrics_path = f"{base_dir}/metrics_summary-filter{filter}.csv"
        output_path = f"{base_dir}/metrics_summary-filter{filter}-pep_plddt.csv"

        new_metrics = []
        new_columns = ['complex', 'seed', 'id', 'pep_plddt', 'gpde', 'ptm', 'iptm', 'ranking_score', 'scRMSD']
        with open(metrics_path, 'r') as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if i >= 1:
                parts = line.strip().split(',')
                complex = parts[0]
                seed = parts[1]
                id = parts[2]
                gpde = float(parts[4])
                ptm = float(parts[5])
                iptm = float(parts[6])
                ranking_score = float(parts[7])
                scRMSD = float(parts[8])

                json_path = f"{base_dir}/{complex}/{seed}/predictions/{complex}_summary_confidence_sample_{id}.json"
                json_file = open(json_path, 'r')
                data = json.load(json_file)
                pep_plddt = round(data['chain_plddt'][1], 4)
                # print(f"{complex}, {seed}, {id}, {pep_plddt}")
                new_metrics.append([complex, seed, id, pep_plddt, gpde, ptm, iptm, ranking_score, scRMSD])
        result = pd.DataFrame(new_metrics, columns=new_columns)
        result.to_csv(output_path, index=False)

### 读取不同csv

In [ ]:
# import os
# import pandas as pd

# # 将scRMSD小于2.5的结果保存在新的csv文件中
# filter = 12
# path = "10000-0.1T"
# cutoff = 2

# with open('./datasets/PepSet_3per_length/PDB.list', 'r') as f:
#     pdbs = [line.strip() for line in f if line.strip()]
# # print(pdbs)
# success_sum = 0
# overall_sum = 0
# for pdb in pdbs:
#     fil_df = pd.read_csv(f'./Protenix/{path}/{pdb}/predict_output-filter{filter}/metrics_filtered_scRMSD_le{cutoff}-filter{filter}.csv')
#     df = pd.read_csv(f'./Protenix/{path}/{pdb}/predict_output-filter{filter}/metrics_summary-filter{filter}.csv')
#     # 筛选fil_df中plddt大于85，iptm大于0.7的行中complex不重复的个数，将其保存在一个字典中，key为pdb，value为个数
#     count = fil_df[(fil_df['plddt'] > 85) & (fil_df['iptm'] > 0.7)]['complex'].nunique()
#     oricount = df['complex'].nunique()
#     success_sum += count
#     overall_sum += oricount
#     print(f"{pdb}: {count:>2}, decoys before filtering: {oricount:>2}, success rate(%): {count/oricount*100:.2f}")
# overall_success_rate = success_sum / overall_sum if overall_sum > 0 else 0
# print(f"Overall success rate: {overall_success_rate:.2%}")

5onp:  3, decoys before filtering: 10, success rate(%): 30.00
2aij:  6, decoys before filtering:  8, success rate(%): 75.00
5e33:  7, decoys before filtering:  8, success rate(%): 87.50
1f8h:  0, decoys before filtering: 10, success rate(%): 0.00
4fii:  8, decoys before filtering: 10, success rate(%): 80.00
5njc:  0, decoys before filtering: 10, success rate(%): 0.00
1czy:  1, decoys before filtering: 10, success rate(%): 10.00
3wgx:  9, decoys before filtering: 10, success rate(%): 90.00
4leb:  1, decoys before filtering: 10, success rate(%): 10.00
1w80:  0, decoys before filtering: 10, success rate(%): 0.00
4c5a:  0, decoys before filtering: 10, success rate(%): 0.00
2e4h:  0, decoys before filtering: 10, success rate(%): 0.00
5dai: 10, decoys before filtering: 10, success rate(%): 100.00
4pge:  9, decoys before filtering: 10, success rate(%): 90.00
2khh:  0, decoys before filtering: 10, success rate(%): 0.00
4pd1:  9, decoys before filtering: 10, success rate(%): 90.00
6b27:  8, dec

In [9]:
import os
import pandas as pd

cutoffs = [1, 1.5, 2, 2.5]
filters = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]

pep_plddt_cutoff = 0.7
iptm_cutoff = 0.7

str_plddt_cutoff = "0" + str(int(100 * pep_plddt_cutoff))
str_iptm_cutoff = "0" + str(int(100 * iptm_cutoff))
with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet_3per_length/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

pdb_w = max(4, max(len(p) for p in pdbs))
col_w = 8
block_w = col_w * len(cutoffs) + (len(cutoffs) - 1)

def format_block(values):
    return " ".join([str(v).rjust(col_w) for v in values])

header1 = f"{'PDB':<{pdb_w}}  " + " | ".join([filter.center(block_w) for filter in filters])
header2 = f"{'':<{pdb_w}}  " + " | ".join([format_block([f"{c:g}" for c in cutoffs]) for _ in filters])

print(header1)
print(header2)

csv_rows = []
csv_header = ["pdb"] + [f"{filter}_cutoff_{c:g}" for filter in filters for c in cutoffs]
csv_rows.append(csv_header)

for pdb in pdbs:
    filter_blocks = []
    row = [pdb]
    for filter in filters:
        if filter in ["6", "7", "9", "10"]:
            df_path = f'./Protenix/2000-0.1T/{pdb}/predict_output-filter{filter}/metrics_summary-filter{filter}-pep_plddt.csv'
            # print(df_path)
        elif filter in ["8"]:
            df_path = f'./Protenix/2000-0.2T/{pdb}/predict_output-filter{filter}/metrics_summary-filter{filter}-pep_plddt.csv'
        elif filter in ["1", "11", "12"]:
            df_path = f'./Protenix/10000-0.1T/{pdb}/predict_output-filter{filter}/metrics_summary-filter{filter}-pep_plddt.csv'
        elif filter in ["2", "3", "4", "5"]:
            df_path = f'./Protenix/11000-0.1_0.2T/{pdb}/predict_output-filter{filter}/metrics_summary-filter{filter}-pep_plddt.csv'
        if not os.path.exists(df_path):
            filter_blocks.append(format_block(["NA"] * len(cutoffs)))
            row.extend(["NA"] * len(cutoffs))
            continue
        df = pd.read_csv(df_path)
        oricount = df['complex'].nunique()
        rates = []
        for cutoff in cutoffs:
            count = df[(df['pep_plddt'] > pep_plddt_cutoff) & (df['iptm'] > iptm_cutoff) & (df['scRMSD'] < cutoff)]['complex'].nunique()
            rate = count / oricount * 100
            rate_str = f"{rate:.2f}"
            rates.append(rate_str)
            row.append(rate_str)
        filter_blocks.append(format_block(rates))
    csv_rows.append(row)
    print(f"{pdb:<{pdb_w}}  " + " | ".join(filter_blocks))

df_out = pd.DataFrame(csv_rows[1:], columns=csv_rows[0])
mean_series = df_out.drop(columns=['pdb']).replace('NA', pd.NA).astype(float).mean()
mean_row = ['MEAN'] + [f"{mean_series[col]:.2f}" if pd.notna(mean_series[col]) else "NA" for col in df_out.columns[1:]]
csv_rows.append(mean_row)
print(f"{'MEAN':<{pdb_w}}  " + " | ".join([format_block(mean_row[1 + i * len(cutoffs):1 + (i + 1) * len(cutoffs)]) for i in range(len(filters))]))
csv_output_path = f'./Protenix/success_rates_by_plddt{str_plddt_cutoff}_iptm{str_iptm_cutoff}.csv'
pd.DataFrame(csv_rows[1:], columns=csv_rows[0]).to_csv(csv_output_path, index=False)
print(f"CSV saved to: {csv_output_path}")

PDB                    1                  |                  2                  |                  3                  |                  4                  |                  5                  |                  6                  |                  7                  |                  8                  |                  9                  |                  10                 |                  11                 |                  12                
             1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5
5onp      0.00     0.00     0.00    10.00 |     0.00     0.00     0.00    20.00 

# 对PepSet结构预测后的通过的结果进行三次LigandMPNN设计，并对设计结果进行筛选与预测。计算相较于设计前的预测结构的scRMSD

In [6]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer


with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB_pep_plddt.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]

pdbparser = PDBParser(QUIET=True)
mmcifparser = FastMMCIFParser(QUIET=True)

for ligandmpnn_seed in ['ligandmpnn_seed42', 'ligandmpnn_seed43', 'ligandmpnn_seed44']:
    for pdb in pdbs:
        base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/{pdb}/{ligandmpnn_seed}"    # 这里是对ligandMPNN在随机数种子为42的结果进行处理
        original_pdb_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/dimer_filtered/{pdb}.pdb'
        if not os.path.exists(original_pdb_path):
            raise FileNotFoundError(f"Original PDB file not found: {original_pdb_path}")
        
        structure_ref = pdbparser.get_structure('ref', original_pdb_path)
        chain_L_ref = structure_ref[0]['B']
        ref_atoms = [atom for atom in chain_L_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
    
        rows = []
    
        for root, dirs, files in os.walk(base_dir):
            seed = root.split("/")[-2]
            sample_name = root.split("/")[-3]
    
            candidates = []
            if not root.endswith("predictions"):
                continue
            
            metrics_by_id = {}
    
            for file in files:
                if file.endswith(".json"):
                    sample_id = file.split("_")[-1].split(".")[0]
                    json_path = os.path.join(root, file)
                    if not os.path.exists(json_path):
                        continue
                    df = pd.read_json(json_path)
                    metrics_by_id.setdefault(sample_id, {}).update(
                        {
                            "pep_plddt": round(float(df["chain_plddt"].iloc[1]), 4),
                            "gpde": round(float(df["gpde"].iloc[0]), 4),
                            "ptm": round(float(df["ptm"].iloc[0]), 4),
                            "iptm": round(float(df["iptm"].iloc[0]), 4),
                            "ranking_score": round(float(df["ranking_score"].iloc[0]), 4),
                        }
                    )
    
                elif file.endswith(".cif"):
                    sample_id = file.split("_")[-1].split(".")[0]
                    cif_path = os.path.join(root, file)
                    if not os.path.exists(cif_path):
                        continue
                    structure_pred = mmcifparser.get_structure('pred', cif_path)
    
                    # 预测结构中的 L 链（原逻辑为 'B'）
                    try:
                        chain_L_pred = structure_pred[0]['B']
                    except KeyError:
                        # 若不存在 B 链，跳过
                        continue
                    
                    # 参考结构的蛋白链（选取第一个不为 'B' 的链）
                    align_chain_ref = None
                    for ch in structure_ref[0]:
                        if ch.id != 'B':
                            align_chain_ref = ch
                            break
                    if align_chain_ref is None:
                        continue
                    
                    # 预测结构的蛋白链（选取第一个不为预测 L 链的链）
                    align_chain_pred = None
                    for ch in structure_pred[0]:
                        if ch.id != chain_L_pred.id:
                            align_chain_pred = ch
                            break
                    if align_chain_pred is None:
                        continue
                    
                    # 取用于对齐的蛋白链骨架原子
                    align_ref_atoms = [atom for atom in align_chain_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                    align_pred_atoms = [atom for atom in align_chain_pred.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                    if len(align_ref_atoms) == 0 or len(align_pred_atoms) == 0:
                        raise ValueError("No backbone atoms found for alignment.")
                    if len(align_ref_atoms) != len(align_pred_atoms):
                        # 原子数不一致，无法对齐，跳过
                        raise ValueError("Backbone atom count mismatch for alignment.")
    
                    # 计算对齐变换（基于蛋白链）
                    sup = Superimposer()
                    try:
                        sup.set_atoms(align_ref_atoms, align_pred_atoms)
                    except Exception:
                        raise ValueError("Failed to set atoms for superimposer.")
                    rot, tran = sup.rotran
    
                    # 在该变换下计算 L 链骨架的 RMSD
                    ref_L_atoms = [atom for atom in chain_L_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                    pred_L_atoms = [atom for atom in chain_L_pred.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]
                    if len(ref_L_atoms) == 0 or len(pred_L_atoms) == 0:
                        raise ValueError("No backbone atoms found for L chain.")
                    if len(ref_L_atoms) != len(pred_L_atoms):
                        raise ValueError("Backbone atom count mismatch for L chain.")
    
                    ref_coords = np.array([a.get_coord() for a in ref_L_atoms], dtype=float)
                    pred_coords = np.array([a.get_coord() for a in pred_L_atoms], dtype=float)
                    pred_coords_aligned = pred_coords @ rot + tran
    
                    diffs = ref_coords - pred_coords_aligned
                    scRMSD = float(np.sqrt(np.mean(np.sum(diffs**2, axis=1))))
                    scRMSD = round(scRMSD, 4)
                    metrics_by_id.setdefault(sample_id, {})["scRMSD"] = scRMSD
    
            for sample_id, data in metrics_by_id.items():
                required_keys = {"pep_plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"}
                if not required_keys.issubset(data):
                    continue
                candidates.append(
                    (
                        sample_name,
                        seed,
                        sample_id,
                        data["pep_plddt"],
                        data["gpde"],
                        data["ptm"],
                        data["iptm"],
                        data["ranking_score"],
                        data["scRMSD"],
                    )
                )
    
            # flatten all candidate rows across directories
            rows.extend(candidates)
    
        result = pd.DataFrame(rows, columns=["complex", "seed", "id", "pep_plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"])
        result.to_csv(f"{base_dir}/metrics_summary-pep_plddt.csv", index=False)
    # result


    # # 过滤scRMSD大于2和2.5的结果，保存到新的df中
    # result_filtered = result[result['scRMSD'] <= 2.0].reset_index(drop=True)
    # result_filtered.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le2.csv", index=False)
    # result_filtered25 = result[result['scRMSD'] <= 2.5].reset_index(drop=True)
    # result_filtered25.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le2.5.csv", index=False)

In [8]:
with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]


for pdb in pdbs:
    for seed in ['ligandmpnn_seed42', 'ligandmpnn_seed43', 'ligandmpnn_seed44']:
        df = pd.read_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_summary.csv')
        result_filtered15 = df[df['scRMSD'] <= 1.5].reset_index(drop=True)
        result_filtered15.to_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_filtered_scRMSD_le1.5.csv', index=False)
        result_filtered10 = df[df['scRMSD'] <= 1.0].reset_index(drop=True)
        result_filtered10.to_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_filtered_scRMSD_le1.csv', index=False)

In [14]:
import os
import pandas as pd

cutoff = 1.5
seed = "ligandmpnn_seed44"


with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]
# print(pdbs)
success_sum = 0
overall_sum = 0
for pdb in pdbs:
    fil_df = pd.read_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_filtered_scRMSD_le{cutoff}.csv')
    df = pd.read_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_summary.csv')
    # 筛选fil_df中plddt大于85，iptm大于0.7的行中complex不重复的个数，将其保存在一个字典中，key为pdb，value为个数
    count = fil_df[(fil_df['plddt'] > 85) & (fil_df['iptm'] > 0.7)]['complex'].nunique()
    oricount = df['complex'].nunique()
    success_sum += count
    overall_sum += oricount
    print(f"{pdb}: {count:>2}, decoys before filtering: {oricount:>2}, success rate(%): {count/oricount*100:.2f}")
overall_success_rate = success_sum / overall_sum if overall_sum > 0 else 0
print(f"Overall success rate: {overall_success_rate:.2%}")

1czy_sample_1:  0, decoys before filtering: 10, success rate(%): 0.00
1f47_sample_2:  9, decoys before filtering: 10, success rate(%): 90.00
1f8h_sample_3:  2, decoys before filtering: 10, success rate(%): 20.00
1j2x_sample_4:  3, decoys before filtering: 10, success rate(%): 30.00
1jd5_sample_4:  9, decoys before filtering: 10, success rate(%): 90.00
1lb6_sample_4: 10, decoys before filtering: 10, success rate(%): 100.00
1nrl_sample_1:  7, decoys before filtering: 10, success rate(%): 70.00
1r17_sample_1:  9, decoys before filtering: 10, success rate(%): 90.00
1rxz_sample_3:  8, decoys before filtering: 10, success rate(%): 80.00
1t74_sample_3:  5, decoys before filtering: 10, success rate(%): 50.00
1v1t_sample_1:  8, decoys before filtering: 10, success rate(%): 80.00
1w80_sample_0:  0, decoys before filtering: 10, success rate(%): 0.00
1ywi_sample_3:  4, decoys before filtering: 10, success rate(%): 40.00
1ywo_sample_2:  3, decoys before filtering: 10, success rate(%): 30.00
1yy6_sa

# 检查每一个结构预测的15个结构的序列是否json文件的序列保持一致

In [12]:
# 检查/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer下每一个pdb的ligandmpnn_seed目录中每一个job name的所有seed的预测结果的B链是否和pred.json的多肽链一致
import os
import json
from Bio.PDB import PDBParser, FastMMCIFParser
from Bio.SeqUtils import seq1

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

mmcifparser = FastMMCIFParser(QUIET=True)
for pdb in pdbs:
    pep_base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/{pdb}"
    seed_dirs = [d for d in os.listdir(pep_base_dir) if d.startswith('ligandmpnn_seed44')]
    for seed in seed_dirs:
        pred_json_path = f'{pep_base_dir}/{seed}/pred.json'
        if not os.path.exists(pred_json_path):
            print(f"pred.json not found for {pdb} {seed}")
            continue
        with open(pred_json_path, 'r') as f:
            jobs = json.load(f)
        updated_jobs = []
        for job in jobs:
           job_name = job['name']
           for protenix_seed in ['seed_42', 'seed_43', 'seed_44']:
                cif_path = f'{pep_base_dir}/{seed}/{job_name}/{protenix_seed}/predictions/{job_name}_sample_0.cif'
                if not os.path.exists(cif_path):
                   continue
                structure_pred = mmcifparser.get_structure('pred', cif_path)
                chain_B_pred = structure_pred[0]['B']
                peptide_sequence = ''.join([seq1(residue.get_resname()) for residue in chain_B_pred.get_residues() if residue.get_id()[0] == ' '])
                if peptide_sequence != job['sequences'][1]['proteinChain']['sequence']:
                    print(f"Sequence mismatch for {pdb} {seed} {job_name} {protenix_seed}: pred.json sequence: {job['sequences'][1]['proteinChain']['sequence']}, CIF sequence: {peptide_sequence}")
                else:
                    print(f"Sequence match for {pdb} {seed} {job_name} {protenix_seed}:pred.json sequence: {job['sequences'][1]['proteinChain']['sequence']}, CIF sequence: {peptide_sequence} ")

Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_42:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_43:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_44:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_42:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_43:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_44:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_3 seed_42:pred.json sequence: ATRASST, CIF sequence: ATRASST 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_3 seed_43:pred.json sequence: ATRASST, CIF seq